# Spark at Scale: One Year of Taxi Trips

In the [pandas notebook](pandas_limits.ipynb) one year of taxi trips did not fit in memory. Here we analyze the **same year with Spark**, on **Databricks**.

**The data:** every taxi trip in New York City, published by the [NYC Taxi and Limousine Commission](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page) as one Parquet file per month. In 2024 there were about **3.4 million trips per month**, **41 million** in the whole year. Each trip has 19 columns: pickup and drop-off time, distance, fare, tip, payment type, and more (see the [data dictionary](https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf)).

Run the cells from top to bottom (**Shift + Enter**).

## 0. Setup: a place for the files

On Databricks, files are stored in a **volume** (a folder managed by Databricks). The next cell creates one called `taxi_data`, if it does not exist yet.

In [ ]:
import os
import time
import urllib.request

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.taxi_data")
DATA_DIR = "/Volumes/workspace/default/taxi_data"

### Download the 12 months of 2024
About 650 MB in total: it takes a few minutes the first time. If you run the cell again, files already downloaded are skipped.

In [ ]:
BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"

start = time.perf_counter()
for month in range(1, 13):
    name = f"yellow_tripdata_2024-{month:02d}.parquet"
    path = f"{DATA_DIR}/{name}"
    if not os.path.exists(path):
        urllib.request.urlretrieve(f"{BASE_URL}/{name}", path)
    print(f"{name}: {os.path.getsize(path) / 1024 ** 2:.0f} MB")
print(f"done in {time.perf_counter() - start:.0f} s")

## 1. Read the whole year

One line reads all 12 files: the `*` means "every file whose name matches". The cell returns immediately, because Spark is **lazy**: it only looks at the column names, it does not read the data yet.

In [ ]:
trips = spark.read.parquet(f"{DATA_DIR}/yellow_tripdata_2024-*.parquet")
trips.printSchema()

In [ ]:
start = time.perf_counter()
print(f"trips in 2024: {trips.count():,} | time: {time.perf_counter() - start:.1f} s")

## 2. The same question as in pandas, on the whole year

Average tip by payment type, on **41 million** trips. In pandas we could not even load the data; here it takes a few seconds.

In [ ]:
start = time.perf_counter()
tips = (trips
    .groupBy("payment_type")
    .agg(F.count("*").alias("trips"), F.round(F.avg("tip_amount"), 2).alias("avg_tip"))
    .orderBy("payment_type"))
tips.show()
print(f"time: {time.perf_counter() - start:.1f} s")

Payment types: 1 = credit card, 2 = cash, 3 = no charge, 4 = dispute, 5 = unknown, 0 = flex fare. Tips paid in cash are not recorded, so the average for cash is (almost) 0.

## 3. Why is Spark so fast? It reads only what it needs

Parquet is a **columnar** format: each column is stored separately in the file. Spark looks at the query *before* running it and reads **only the columns it needs**. Our query uses 2 columns out of 19.

`explain()` shows the plan. Look at the line that reads the Parquet files (it contains `Scan parquet`): only `payment_type` and `tip_amount` are listed, and **ReadSchema** contains only these two columns.

In [ ]:
tips.explain()

Spark can also skip data with a **filter**. For example, only trips longer than 100 miles: look at **PushedFilters** (or **DataFilters**) in the plan. The filter is sent to the file reader, which skips the parts of the files that cannot contain such trips.

In [ ]:
long_trips = trips.filter(F.col("trip_distance") > 100).select("tpep_pickup_datetime", "trip_distance", "total_amount")
long_trips.explain()
print(f"trips longer than 100 miles: {long_trips.count():,}")

## 4. One billion rows

The taxi year is "only" 41 million rows. What about **one billion**? `spark.range` creates a DataFrame with the numbers from 0 to 999,999,999, and we compute some statistics on it.

In pandas, just one column of one billion numbers would need **8 GB** of memory, before doing anything with it. Spark never builds the whole table in one place: every machine generates and processes its own part.

In [ ]:
start = time.perf_counter()
billion = spark.range(1_000_000_000)

(billion
    .withColumn("is_even", (F.col("id") % 2 == 0))
    .groupBy("is_even")
    .agg(F.count("*").alias("rows"), F.sum("id").alias("sum"))
    .show())
print(f"time: {time.perf_counter() - start:.1f} s")

## 5. Your turn

Answer these questions on the **whole year**. Some trips have wrong dates (for example in 2002 or 2009): keep only the trips of 2024 where it matters.

1. What is the **busiest hour** of the day? (hint: `F.hour("tpep_pickup_datetime")`)
2. Which **month** has the most trips? (hint: `F.month(...)`)
3. How many trips **paid by credit card** have a tip of more than 20% of the fare (`fare_amount`)?
4. **Airport trips**: how many trips per month paid an airport fee (`Airport_fee > 0`)?
5. **Data quality**: how many trips have a distance of 0, or a negative `total_amount`? What would you do with them?

In [ ]:
# Question 1

In [ ]:
# Question 2

In [ ]:
# Question 3

In [ ]:
# Question 4

In [ ]:
# Question 5

## 6. When NOT to use Spark

Spark is not always the best choice. Starting a distributed job has a cost (planning, sending tasks to the machines, collecting the results). On **small** data, pandas is often faster.

Let's run the same question on **one month** with both.

In [ ]:
import pandas as pd

path = f"{DATA_DIR}/yellow_tripdata_2024-01.parquet"

start = time.perf_counter()
pdf = pd.read_parquet(path, columns=["payment_type", "tip_amount"])
pdf.groupby("payment_type")["tip_amount"].mean()
print(f"pandas, one month: {time.perf_counter() - start:.2f} s")

start = time.perf_counter()
spark.read.parquet(path).groupBy("payment_type").agg(F.avg("tip_amount")).collect()
print(f"Spark, one month:  {time.perf_counter() - start:.2f} s")

## Summary

- **pandas**: one machine, all the data in memory. Simple and fast, as long as the data fits.
- **Spark**: the data is split across many machines, read lazily, and only the columns and rows a query needs are read. The same code works on 1 million or 1 billion rows.
- **Rule of thumb**: if your data fits comfortably in memory, use pandas. If it does not, or if it will keep growing, use Spark.